In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. PREPARAZIONE DEI DATI
# ==========================================

# Carica i dati (assicurati che il file si chiami esattamente così)
df = pd.read_csv('mpi_benchmark_results.csv')

# Converte la colonna Tempo_Medio in numerico, trasformando eventuali 'ERROR' in NaN
df['Tempo_Medio(s)'] = pd.to_numeric(df['Tempo_Medio(s)'], errors='coerce')

# Ordina il dataframe per consistenza
df = df.sort_values(by=['N_Size', 'Num_Processes'])

# Calcolo delle metriche derivate
# 1. Speedup (Sp = T1 / Tp)
df['Speedup'] = df.groupby('N_Size')['Tempo_Medio(s)'].transform(lambda x: x.iloc[0] / x)

# 2. Efficienza (Ep = Sp / p)
df['Efficiency'] = df['Speedup'] / df['Num_Processes']

# 3. GFLOPS = (2 * N^3) / (Tempo * 10^9)
# La moltiplicazione di matrici N x N richiede 2N^3 operazioni floating point
df['GFLOPS'] = (2 * (df['N_Size'].astype(float) ** 3)) / (df['Tempo_Medio(s)'] * 1e9)

# 4. Metrica di Karp-Flatt (e)
# e = ( (1/Sp) - (1/p) ) / (1 - (1/p))
# Gestiamo il caso p=1 per evitare divisioni per zero
def karp_flatt(row):
    p = row['Num_Processes']
    Sp = row['Speedup']
    if p == 1 or pd.isna(Sp):
        return np.nan
    return ((1 / Sp) - (1 / p)) / (1 - (1 / p))

df['Karp_Flatt'] = df.apply(karp_flatt, axis=1)

# Imposta uno stile pulito per i grafici
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

# ==========================================
# 2. GENERAZIONE GRAFICI STANDARD
# ==========================================

# Plot 1: Tempo di esecuzione (Log Scale)
plt.figure(figsize=(10, 6))
sns.lineplot(data=df, x='Num_Processes', y='Tempo_Medio(s)', hue='N_Size', marker='o', palette='tab10')
plt.yscale('log')
plt.title('Execution Time vs Number of Processes')
plt.xlabel('Number of Processes ($p$)')
plt.ylabel('Execution Time (s) [Log Scale]')
plt.legend(title='Matrix Size ($N$)', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('plot_1_execution_time.png', dpi=300)
plt.close()

# Plot 2: Speedup
plt.figure(figsize=(10, 6))
sns.lineplot(data=df, x='Num_Processes', y='Speedup', hue='N_Size', marker='o', palette='tab10')
plt.plot([1, 32], [1, 32], 'k--', label='Ideal Speedup', alpha=0.7)
plt.title('Speedup vs Number of Processes')
plt.xlabel('Number of Processes ($p$)')
plt.ylabel('Speedup ($S_p$)')
plt.legend(title='Matrix Size ($N$)', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('plot_2_speedup.png', dpi=300)
plt.close()

# Plot 3: Efficienza
plt.figure(figsize=(10, 6))
sns.lineplot(data=df, x='Num_Processes', y='Efficiency', hue='N_Size', marker='o', palette='tab10')
plt.axhline(1.0, color='k', linestyle='--', alpha=0.5, label='Ideal Efficiency')
plt.title('Parallel Efficiency vs Number of Processes')
plt.xlabel('Number of Processes ($p$)')
plt.ylabel('Efficiency ($E_p$)')
plt.legend(title='Matrix Size ($N$)', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('plot_3_efficiency.png', dpi=300)
plt.close()

# ==========================================
# 3. GENERAZIONE NUOVI GRAFICI AVANZATI (HPC)
# ==========================================

# Plot 4: GFLOPS (Performance Assoluta)
plt.figure(figsize=(10, 6))
sns.lineplot(data=df, x='Num_Processes', y='GFLOPS', hue='N_Size', marker='s', palette='tab10')
plt.title('Computational Throughput (GFLOPS) vs Processes')
plt.xlabel('Number of Processes ($p$)')
plt.ylabel('Performance (GFLOPS)')
plt.legend(title='Matrix Size ($N$)', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('plot_4_gflops.png', dpi=300)
plt.close()

# Plot 5: Esecuzione vs N (Log-Log) per verificare complessità O(N^3)
plt.figure(figsize=(10, 6))
sns.lineplot(data=df, x='N_Size', y='Tempo_Medio(s)', hue='Num_Processes', marker='^', palette='tab10')
plt.xscale('log')
plt.yscale('log')
plt.title('Execution Time vs Problem Size ($N$)')
plt.xlabel('Matrix Dimension ($N$) [Log Scale]')
plt.ylabel('Execution Time (s) [Log Scale]')
plt.legend(title='Processes ($p$)', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('plot_5_time_vs_n.png', dpi=300)
plt.close()

# Plot 6: Heatmap dell'Efficienza
plt.figure(figsize=(10, 8))
pivot_eff = df.pivot(index="N_Size", columns="Num_Processes", values="Efficiency")
sns.heatmap(pivot_eff, annot=True, fmt=".2f", cmap="RdYlGn", cbar_kws={'label': 'Efficiency'})
plt.title('Efficiency Heatmap ($N$ vs $p$)')
plt.xlabel('Number of Processes ($p$)')
plt.ylabel('Matrix Size ($N$)')
plt.gca().invert_yaxis() # Per avere le matrici piccole in basso
plt.tight_layout()
plt.savefig('plot_6_efficiency_heatmap.png', dpi=300)
plt.close()

# Plot 7: Metrica di Karp-Flatt (Frazione Seriale / Overhead)
plt.figure(figsize=(10, 6))
sns.lineplot(data=df.dropna(subset=['Karp_Flatt']), x='Num_Processes', y='Karp_Flatt', hue='N_Size', marker='D', palette='tab10')
plt.title('Karp-Flatt Metric (Serial Fraction & Communication Overhead)')
plt.xlabel('Number of Processes ($p$)')
plt.ylabel('Karp-Flatt Metric ($e$)')
plt.legend(title='Matrix Size ($N$)', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('plot_7_karp_flatt.png', dpi=300)
plt.close()

print("Generazione dei 7 grafici completata con successo! I file PNG sono stati salvati.")

Generazione dei 7 grafici completata con successo! I file PNG sono stati salvati.
